# Initialization

## Libraries

In [1]:
%load_ext autoreload
%autoreload 2

from skopt import gp_minimize
from skopt.space import Real, Categorical
from skopt.learning import GaussianProcessRegressor
from skopt.learning.gaussian_process.kernels import Matern, RBF

import warnings
import numpy as np
import pandas as pd
from pathlib import Path

## Directories

In [2]:
data_dir = Path("data")
Path.mkdir(data_dir, exist_ok=True)

plot_dir = Path("plots")
Path.mkdir(plot_dir, exist_ok=True)

log_dir = Path("logs")
Path.mkdir(log_dir, exist_ok=True)

## Data reading

In [3]:
data_file = "Dataset_SL.xlsx"

### Define experimental data

In [4]:
# 1. Define the experimental data
experiment_data = pd.read_excel(data_dir / data_file, sheet_name="Datasheet")

In [5]:
# make the other columns as floats
experiment_data = experiment_data.astype(
    {
        "salt_concentration": float,
        "water_to_cement_ratio": float,
        "antisettling_concentration": float,
        "E_d": float,
        "KPI": float,
    }
)

# get optimization round for later use in saving
opt_round = experiment_data["opt_round"].iloc[-1]

In [ ]:
experiment_data

# Search space

### Category encoding

In [ ]:
cat_order = [
    "Al2(SO4)3",
    "CaCl2",
    "CuSO4",
    "K2CO3",
    "KAl(SO4)2",
    "LiCl",
    "Mg(NO3)2",
    "MgCl2",
    "MgSO4",
    "SrBr2",
    "Zn(NO3)2",
]

cat_mapping = {name: i for i, name in enumerate(cat_order)}

df_enc = experiment_data.copy()
df_enc["category_encoded"] = experiment_data["salt"].map(cat_mapping)
df_enc

In [8]:
search_space = [
    Categorical(cat_order, name="salt"),  # salt names
    Real(0.1, 0.9, name="salt_concentration"),
    Real(0.7, 1.5, name="water_to_cement_ratio"),
    Real(0.0, 3.0, name="antisettling_concentration"),
]

# Optimization

In [9]:
salt_dummies = pd.get_dummies(df_enc["salt"], prefix="salt").reindex(
    columns=[f"salt_{cat}" for cat in cat_order], fill_value=0
)

In [10]:
X = pd.concat(
    [
        salt_dummies,
        df_enc[
            [
                "salt_concentration",
                "water_to_cement_ratio",
                "antisettling_concentration",
            ]
        ],
    ],
    axis=1,
).to_numpy()
y_energy = df_enc["E_d"].values
y_kpi = df_enc["KPI"].values

In [ ]:
salt_dummies

## Gaussian Process

In [12]:
def fit_gp_models(X, y, kernel):
    """Fits Gaussian Process models to the given experimental data."""  # noqa

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        alpha=1e-3,
    )

    gp.fit(X, y)

    return gp

### Matern kernel

In [17]:
matern_kernel = Matern(
    length_scale=5e-3, length_scale_bounds=(1e-8, 10.0), nu=2.5
)  # noqa

In [18]:
gp_energy_matern = fit_gp_models(X, -y_energy, matern_kernel)

In [19]:
gp_kpi_matern = fit_gp_models(X, y_kpi, matern_kernel)

### RBF kernel

In [20]:
rbf_kernel = RBF(length_scale=5e-3, length_scale_bounds=(1e-8, 10.0))

In [21]:
gp_energy_rbf = fit_gp_models(X, -y_energy, rbf_kernel)

In [22]:
gp_kpi_rbf = fit_gp_models(X, y_kpi, rbf_kernel)

# Bayesian Optimization

In [23]:
def encode_input(x):
    # x[0] is salt name (e.g., "MgSO4")
    salt_vector = np.zeros(len(cat_mapping))
    salt_index = cat_mapping[x[0]]
    salt_vector[salt_index] = 1

    # concatenate with continuous features
    return np.concatenate([salt_vector, np.array(x[1:])])

In [24]:
def suggest_new_samples(
    gp_model: GaussianProcessRegressor,
    kernel: Matern | RBF,
    obj_func: str,
    print_points: bool = False,
    verbose: bool = False,
    mute_warnings: bool = True,
):
    """Suggests new samples based on the trained GP model using different acquisition functions."""  # noqa

    acq_functions = ["EI", "PI", "LCB_low", "LCB_med", "LCB_high"]
    kappa_values = {
        "LCB_low": 1.0,
        "LCB_med": 4.0,
        "LCB_high": 10.0,
    }
    new_samples = []

    with warnings.catch_warnings():
        if mute_warnings:
            print("Warnings are muted!")
            warnings.simplefilter("ignore")

        for i, acquisition in enumerate(acq_functions):
            res = gp_minimize(
                lambda x: gp_model.predict([encode_input(x)])[
                    0
                ],  # Optimize our surrogate model # noqa
                dimensions=search_space,  # pass our search space
                base_estimator=GaussianProcessRegressor(
                    kernel=kernel,
                    normalize_y=True,
                    n_restarts_optimizer=10,
                    alpha=1e-3,
                ),
                acq_func=(
                    "LCB"
                    if acquisition in ["LCB_low", "LCB_med", "LCB_high"]
                    else acquisition
                ),  # define the acquisition function
                kappa=kappa_values.get(
                    acquisition
                ),  # define the custom k value if acquisition if "LCB" # noqa
                xi=0.05,
                n_calls=40,
                verbose=verbose,
                n_jobs=6,
            )

            # Convert numerical salt encoding back to categorical
            suggested = res.x
            if print_points:
                print(suggested)

            suggested.append(str(gp_model.kernel).split("(")[0])
            suggested.append(str(acquisition))
            suggested.append(str(obj_func))

            # save suggested samples in a list
            new_samples.append(suggested)

    # create a new dataframe with suggested samples
    columns = [
        "salt",
        "salt_concentration",
        "water_to_cement_ratio",
        "antisettling_concentration",
        "kernel",
        "acquisition",
        "obj_func",
    ]

    df = pd.DataFrame(new_samples, columns=columns)

    # round float values to 3rd decimal place
    df[df.select_dtypes(include="float").columns] = df.select_dtypes(
        include="float"
    ).round(3)

    return df

### Generate 10 new samples for Energy Density optimization

In [ ]:
# Matérn kernel
new_samples_energy_matern = suggest_new_samples(
    gp_energy_matern,
    matern_kernel,
    obj_func="E_d",
    verbose=True,
    mute_warnings=True,
)

In [ ]:
new_samples_energy_matern

In [ ]:
# RBF kernel
new_samples_energy_rbf = suggest_new_samples(
    gp_energy_rbf, rbf_kernel, obj_func="E_d", verbose=True, mute_warnings=True
)

In [ ]:
new_samples_energy_rbf

### Generate 10 new samples for economic KPI optimization

In [ ]:
# Matérn kernel
new_samples_kpi_matern = suggest_new_samples(
    gp_kpi_matern,
    matern_kernel,
    obj_func="KPI",
    verbose=True,
    mute_warnings=True,  # noqa
)

In [ ]:
new_samples_kpi_matern

In [ ]:
# RBF kernel
new_samples_kpi_rbf = suggest_new_samples(
    gp_kpi_rbf, rbf_kernel, obj_func="KPI", verbose=True, mute_warnings=True
)  # noqa

In [ ]:
new_samples_kpi_rbf

## Saving the new suggestions in the original excel sheet

In [ ]:
# Combine sets of new samples
new_samples = pd.concat(
    [
        new_samples_energy_matern,
        new_samples_energy_rbf,
        new_samples_kpi_matern,
        new_samples_kpi_rbf,
    ],
    ignore_index=True,
)

new_samples.insert(0, "opt_round", opt_round + 1)  # add optimization round

# Read the existing Excel sheet into a DataFrame
experiment_data = pd.read_excel(data_dir / data_file, sheet_name="Datasheet")
# Append the new data to the existing DataFrame
combined_data = pd.concat(
    [experiment_data, new_samples], ignore_index=True, axis=0
)  # noqa

# Write the updated DataFrame back to the same Excel sheet
with pd.ExcelWriter(
    data_dir / data_file, engine="openpyxl", mode="a", if_sheet_exists="replace"  # noqa
) as writer:
    combined_data.to_excel(writer, sheet_name="Datasheet", index=False)

print(
    "New batch of 20 samples suggested. Please conduct experiments and update the dataset."  # noqa
)